# FinQA GRPO 对比实验：Base vs v3 Program SFT

本 notebook 面向 **FinQA / ConvFinQA 可验证数值推理** 的 GRPO 对比实验。

当前只保留两条训练线：

- `base + GRPO`
- `/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged + GRPO`

训练不再依赖重新执行整个 notebook。数据处理完成后，直接运行生成好的 `run_xxx.sh` 即可开始训练。


## 实验原则

本 notebook 只负责两类事情：

1. 构造并审计 strict Program GRPO 数据
2. 给出可直接运行的 shell 脚本与 benchmark 命令

当前不再训练 `cot_pot_program_mixed + GRPO`，因为远端没有可直接加载的 `program_mixed_merged` 本地模型目录。


In [8]:
import json
import math
import os
import random
import re
import subprocess
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda devices:', torch.cuda.device_count())
except Exception as exc:
    print('torch import failed:', repr(exc))

try:
    import datasets
    print('datasets:', datasets.__version__)
except Exception as exc:
    print('datasets import failed:', repr(exc))

try:
    import trl
    print('trl:', trl.__version__)
except Exception as exc:
    print('trl import failed. Install/update TRL before GRPO training:', repr(exc))


torch: 2.8.0+cu128
cuda available: True
cuda devices: 1
datasets: 4.8.4
trl: 1.1.0


In [9]:
BASE_MODEL = Path('/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct')
V3_SFT2_PROGRAM_MERGED = Path('/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged')

COMPARE_OUTPUT_ROOT = Path('/root/autodl-tmp/outputs/financial_reasoning_grpo_compare')
TRAINING_SCRIPT = Path('/root/FinQA/training/finqa_program_grpo.py')
BASE_GRPO_SCRIPT = Path('/root/FinQA/run_finqa_program_grpo_base.sh')
V3_GRPO_SCRIPT = Path('/root/FinQA/run_finqa_program_grpo_v3.sh')

if not V3_SFT2_PROGRAM_MERGED.exists():
    raise FileNotFoundError(f'missing v3 merged model: {V3_SFT2_PROGRAM_MERGED}')

EXPERIMENT_SPECS = {
    'base': {
        'label': 'Base model only',
        'policy_init_path': BASE_MODEL,
        'policy_benchmark_path': BASE_MODEL,
        'output_root': COMPARE_OUTPUT_ROOT / 'base',
        'legacy_summary': None,
        'legacy_predictions': None,
    },
    'v3_program_sft': {
        'label': 'v3 strict Program SFT',
        'policy_init_path': V3_SFT2_PROGRAM_MERGED,
        'policy_benchmark_path': V3_SFT2_PROGRAM_MERGED,
        'output_root': COMPARE_OUTPUT_ROOT / 'v3_program_sft',
        'legacy_summary': Path('/root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/sft2_merged_passk/benchmark_summary.csv'),
        'legacy_predictions': Path('/root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/sft2_merged_passk/sft2_predictions.jsonl'),
    },
}

EXPERIMENT_ORDER = ['base', 'v3_program_sft']
DEFAULT_EXPERIMENT_KEY = 'v3_program_sft'
ACTIVE_POLICY_KEY = DEFAULT_EXPERIMENT_KEY
ACTIVE_POLICY = EXPERIMENT_SPECS[ACTIVE_POLICY_KEY]
POLICY_BASE = ACTIVE_POLICY['policy_init_path']
SFT2_MERGED_OUT = ACTIVE_POLICY['policy_benchmark_path']


def experiment_output_root(experiment_key: str) -> Path:
    return EXPERIMENT_SPECS[experiment_key]['output_root']


def grpo_output_dir(experiment_key: str, smoke: bool = False) -> Path:
    name = 'grpo_smoke_lora' if smoke else 'grpo_lora'
    return experiment_output_root(experiment_key) / name


def benchmark_output_dir(experiment_key: str, run_kind: str) -> Path:
    return experiment_output_root(experiment_key) / 'benchmarks' / run_kind


RL_DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning_cot_pot/rl')
FINO1_CACHE_DIR = RL_DATA_ROOT / 'fino1_finqa_path'
FINCOT_CACHE_DIR = RL_DATA_ROOT / 'fincot'
GRPO_TRAIN_FILE = RL_DATA_ROOT / 'train_cot_pot_grpo_mixed.jsonl'
GRPO_VALID_FILE = RL_DATA_ROOT / 'valid_cot_pot_grpo_mixed.jsonl'
GRPO_SMOKE_FILE = RL_DATA_ROOT / 'smoke_cot_pot_grpo_mixed.jsonl'

DPO_DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning_cot_pot/dpo_pairs')
DPO_MIXED_FILE = DPO_DATA_ROOT / 'train_cot_pot_program_dpo.jsonl'
DPO_TRAIN_DIR = DPO_DATA_ROOT / 'train_dir'
DPO_SMOKE_FILE = DPO_DATA_ROOT / 'smoke_cot_pot_program_dpo.jsonl'
DPO_SMOKE_DIR = DPO_DATA_ROOT / 'smoke_dir'

OUTPUT_ROOT = experiment_output_root(ACTIVE_POLICY_KEY)
DPO_OUT = OUTPUT_ROOT / 'dpo_program_lora'
DPO_SMOKE_OUT = OUTPUT_ROOT / 'dpo_program_lora_smoke'
DPO_BENCH_OUT = benchmark_output_dir(ACTIVE_POLICY_KEY, 'dpo_program_passk')
GRPO_OUT = grpo_output_dir(ACTIVE_POLICY_KEY, smoke=False)
GRPO_SMOKE_OUT = grpo_output_dir(ACTIVE_POLICY_KEY, smoke=True)
GRPO_BENCH_OUT = benchmark_output_dir(ACTIVE_POLICY_KEY, 'grpo_passk')

PROGRAM_DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning_cot_pot/program_sft')
FINQA_TRAIN_STRICT = PROGRAM_DATA_ROOT / 'train_finqa_program_strict.jsonl'
SFT2_CONV_STRICT = PROGRAM_DATA_ROOT / 'train_convfinqa_turn_program_strict.jsonl'
SFT2_FINQA_REPLAY = PROGRAM_DATA_ROOT / 'train_program_mixed.jsonl'

if not FINQA_TRAIN_STRICT.exists():
    FINQA_TRAIN_STRICT = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft1_program_strict.jsonl')
if not SFT2_CONV_STRICT.exists():
    SFT2_CONV_STRICT = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_convfinqa_turn_program_strict.jsonl')
if not SFT2_FINQA_REPLAY.exists():
    SFT2_FINQA_REPLAY = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_finqa_replay_program.jsonl')

FINQA_EVAL_FILE = Path('/root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json')
CONVFINQA_EVAL_FILE = Path('/root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json')
DESIGN_DOC = Path('/root/FinQA/run_fingpt_cot_pot.ipynb')

for key in EXPERIMENT_ORDER:
    experiment_output_root(key).mkdir(parents=True, exist_ok=True)

experiment_df = pd.DataFrame([
    {
        'experiment_key': key,
        'label': spec['label'],
        'policy_init_path': str(spec['policy_init_path']),
        'policy_benchmark_path': str(spec['policy_benchmark_path']),
        'output_root': str(spec['output_root']),
    }
    for key, spec in EXPERIMENT_SPECS.items()
])
display(experiment_df)
print('ACTIVE_POLICY_KEY for backward-compatible DPO cells:', ACTIVE_POLICY_KEY)


,experiment_key,label,policy_init_path,policy_benchmark_path,output_root
0,base,Base model only,/root/autodl-tmp/models/qwen/Qwen2___5-7B-Inst...,/root/autodl-tmp/models/qwen/Qwen2___5-7B-Inst...,/root/autodl-tmp/outputs/financial_reasoning_g...
1,v3_program_sft,v3 strict Program SFT,/root/autodl-tmp/outputs/financial_reasoning_v...,/root/autodl-tmp/outputs/financial_reasoning_v...,/root/autodl-tmp/outputs/financial_reasoning_g...
2,cot_pot_program_mixed,CoT + PoT mixed Program SFT,/root/autodl-tmp/outputs/financial_reasoning_c...,/root/autodl-tmp/outputs/financial_reasoning_c...,/root/autodl-tmp/outputs/financial_reasoning_g...


ACTIVE_POLICY_KEY for backward-compatible DPO cells: cot_pot_program_mixed


## 数据集方案

本 notebook 对 **GRPO** 采用单一 strict Program 路线，不再构造 mixed `program_numeric + cot_answer_only` 数据。

保留两条训练线，但 contract 分开：

1. **DPO strict-program pairs**
   - 来源：FinQA / ConvFinQA strict Program ShareGPT 数据。
   - `response_chosen`：gold `Evidence + Program`。
   - `response_rejected`：自动扰动 Program 或结构字段。

2. **GRPO program-only core data**
   - 只使用 FinQA / ConvFinQA strict Program 样本。
   - 所有样本 `reward_profile = program_numeric`。
   - 不引入 Fino1 / FinCoT supplement。

统一字段：

- `input_prompt_raw`
- `reward_profile`
- `source_dataset`
- `gold_program`
- `gold_answer`
- `reference_response`
- `record_id`
- `metadata`

训练时再从 `input_prompt_raw` 动态构造 chat prompt，避免覆盖原始题面。所有 evidence-level reward 默认读取 `input_prompt_raw`，而不是 chat list 的字符串化结果。


## Schema Contract

这一版 GRPO notebook 固定为 **单一 reward profile**：

- `program_numeric_output_schema = Evidence + Program`
- GRPO 所有样本都必须显式携带 `reward_profile = program_numeric`
- 主 benchmark 只报告 strict Program 指标
- 不再训练或评估 `Reasoning + Answer` supplement


In [10]:
PROGRAM_NUMERIC_REQUIRED = ['Evidence:', 'Program:']
PROGRAM_NUMERIC_FORBIDDEN = ['Reasoning:', 'Answer:', 'Normalized Answer:', 'Program: N/A']
REWARD_PROFILES = ['program_numeric']

schema_contract = pd.DataFrame([
    {
        'reward_profile': 'program_numeric',
        'output_schema': 'Evidence + Program',
        'required': ', '.join(PROGRAM_NUMERIC_REQUIRED),
        'forbidden': ', '.join(PROGRAM_NUMERIC_FORBIDDEN),
        'main_benchmark': True,
    },
])
print('active_policy_base:', POLICY_BASE)
print('reward_profiles:', REWARD_PROFILES)
display(schema_contract)


active_policy_base: /root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged
reward_profiles: ['program_numeric']


,reward_profile,output_schema,required,forbidden,main_benchmark
0,program_numeric,Evidence + Program,"Evidence:, Program:","Reasoning:, Answer:, Normalized Answer:, Progr...",True


In [11]:
SEED = 42
MAX_CORE_RL_ROWS = 3500
CORE_CONV_TO_FINQA_RATIO = 2.0

TRAIN_CONV_ROWS = 2333
TRAIN_FINQA_ROWS = 1167
VALID_CONV_ROWS = 307
VALID_FINQA_ROWS = 153
SMOKE_CONV_ROWS = 43
SMOKE_FINQA_ROWS = 21

EXPAND_TO_FULL_BALANCED_CORE = False
FULL_BALANCED_TRAIN_FILE = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced.jsonl')
FULL_BALANCED_SUMMARY_FILE = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced_summary.json')
GRPO_DATA_SUMMARY_FILE = RL_DATA_ROOT / 'grpo_program_only_summary.json'

assert TRAIN_CONV_ROWS + TRAIN_FINQA_ROWS == MAX_CORE_RL_ROWS
assert VALID_CONV_ROWS + VALID_FINQA_ROWS == 460
assert SMOKE_CONV_ROWS + SMOKE_FINQA_ROWS == 64
assert abs(TRAIN_CONV_ROWS / TRAIN_FINQA_ROWS - CORE_CONV_TO_FINQA_RATIO) < 0.01

random.seed(SEED)
print({
    'MAX_CORE_RL_ROWS': MAX_CORE_RL_ROWS,
    'TRAIN_CONV_ROWS': TRAIN_CONV_ROWS,
    'TRAIN_FINQA_ROWS': TRAIN_FINQA_ROWS,
    'VALID_CONV_ROWS': VALID_CONV_ROWS,
    'VALID_FINQA_ROWS': VALID_FINQA_ROWS,
    'SMOKE_CONV_ROWS': SMOKE_CONV_ROWS,
    'SMOKE_FINQA_ROWS': SMOKE_FINQA_ROWS,
    'EXPAND_TO_FULL_BALANCED_CORE': EXPAND_TO_FULL_BALANCED_CORE,
    'FULL_BALANCED_TRAIN_FILE': str(FULL_BALANCED_TRAIN_FILE),
})


{'MAX_CORE_RL_ROWS': 3500, 'TRAIN_CONV_ROWS': 2333, 'TRAIN_FINQA_ROWS': 1167, 'VALID_CONV_ROWS': 307, 'VALID_FINQA_ROWS': 153, 'SMOKE_CONV_ROWS': 43, 'SMOKE_FINQA_ROWS': 21, 'EXPAND_TO_FULL_BALANCED_CORE': False, 'FULL_BALANCED_TRAIN_FILE': '/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced.jsonl'}


In [12]:
def read_jsonl(path: Path, max_rows: Optional[int] = None) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        print('missing:', path)
        return rows
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows is not None and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            count += 1
    return count


def sample_rows(rows: List[Dict[str, Any]], n: Optional[int], seed: int = SEED) -> List[Dict[str, Any]]:
    rows = list(rows)
    rng = random.Random(seed)
    rng.shuffle(rows)
    if n is None or n < 0 or n >= len(rows):
        return rows
    return rows[:n]


def split_train_valid(rows: List[Dict[str, Any]], valid_ratio: float, seed: int = SEED) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    rows = sample_rows(rows, None, seed)
    valid_n = max(1, int(round(len(rows) * valid_ratio))) if rows else 0
    return rows[valid_n:], rows[:valid_n]


def first_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return ''.join(first_text(v) for v in value)
    if isinstance(value, dict):
        for key in ('text', 'content', 'value'):
            if key in value:
                return first_text(value[key])
        return json.dumps(value, ensure_ascii=False)
    return str(value)


In [13]:
ANCHORS = ['Evidence:', 'Reasoning:', 'Program:', 'Answer:', 'Normalized Answer:']
PROGRAM_OPS = ['add', 'subtract', 'multiply', 'divide', 'greater', 'table_max', 'table_min', 'table_sum', 'table_average', 'average', 'sum', 'max', 'min']
NUMBER_RE = re.compile(r'-?\d+(?:,\d{3})*(?:\.\d+)?%?')


def extract_anchor(text: str, anchor: str) -> str:
    text = first_text(text)
    if not text or anchor not in text:
        return ''
    start = text.index(anchor) + len(anchor)
    end = len(text)
    for other in ANCHORS:
        if other == anchor:
            continue
        pos = text.find(other, start)
        if pos >= 0:
            end = min(end, pos)
    return text[start:end].strip()


def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        if 'content' in completion:
            return first_text(completion.get('content'))
        if 'text' in completion:
            return first_text(completion.get('text'))
    if isinstance(completion, list) and completion:
        return completion_text(completion[0])
    return first_text(completion)


def normalize_number(text: str) -> Optional[float]:
    text = first_text(text)
    if not text:
        return None
    matches = NUMBER_RE.findall(text.replace(',', ''))
    if not matches:
        return None
    raw = matches[-1]
    is_percent = raw.endswith('%')
    if is_percent:
        raw = raw[:-1]
    try:
        value = float(raw)
    except ValueError:
        return None
    return value / 100.0 if is_percent else value


def numeric_equal(pred: str, gold: str, abs_tol: float = 1e-4, rel_tol: float = 1e-4) -> bool:
    # Contract: percentages like 10% are normalized to 0.10 before comparison.
    pred_num = normalize_number(pred)
    gold_num = normalize_number(gold)
    if pred_num is None or gold_num is None:
        return first_text(pred).strip().lower() == first_text(gold).strip().lower()
    return abs(pred_num - gold_num) <= max(abs_tol, abs(gold_num) * rel_tol)


def program_ops(program: str) -> List[str]:
    program = first_text(program).lower()
    return [op for op in PROGRAM_OPS if re.search(rf'\b{re.escape(op)}\s*\(', program)]


def prompt_text_from_any(value: Any) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict) and item.get('role') == 'user':
                parts.append(first_text(item.get('content')))
            else:
                parts.append(first_text(item))
        return '\n'.join(x for x in parts if x).strip()
    if isinstance(value, dict):
        return first_text(value.get('content') or value.get('text') or value.get('value'))
    return first_text(value)


In [14]:
from evaluation.evaluate_financial_benchmarks import execute_prediction_program


def infer_source_dataset(row: Dict[str, Any], fallback: str) -> str:
    meta = row.get('metadata') or {}
    if any(key in row for key in ['conversation_id', 'turn_ind', 'sft_mode']):
        return 'convfinqa_turn'
    if any(key in meta for key in ['conversation_id', 'turn_ind', 'sft_mode']):
        return 'convfinqa_turn'
    source_hint = first_text(row.get('source_dataset') or meta.get('source_dataset') or '').lower()
    if 'convfinqa' in source_hint:
        return 'convfinqa_turn'
    if 'finqa' in source_hint:
        return 'finqa'
    return fallback


def sharegpt_to_grpo(row: Dict[str, Any], source_dataset: str) -> Optional[Dict[str, Any]]:
    conv = row.get('conversations') or []
    if len(conv) < 2:
        return None
    input_prompt_raw = conv[0].get('value') or conv[0].get('content') or ''
    target = conv[1].get('value') or conv[1].get('content') or ''
    if not input_prompt_raw or not target:
        return None

    meta = row.get('metadata') or {}
    gold_answer = meta.get('answer_norm') or meta.get('answer_exe') or extract_anchor(target, 'Normalized Answer:') or extract_anchor(target, 'Answer:')
    gold_program = meta.get('program_canonical') or meta.get('program_raw') or extract_anchor(target, 'Program:')
    if gold_answer is None or first_text(gold_answer) == '':
        return None
    if first_text(gold_program) == '':
        return None

    executed_value, _, _ = execute_prediction_program(first_text(gold_program))
    if executed_value is None or not numeric_equal(str(executed_value), first_text(gold_answer)):
        return None

    record_id = first_text(
        row.get('record_id')
        or meta.get('record_id')
        or row.get('id')
        or meta.get('raw_id')
        or row.get('conversation_id')
        or meta.get('conversation_id')
    )
    return {
        'input_prompt_raw': input_prompt_raw,
        'gold_answer': first_text(gold_answer),
        'gold_program': first_text(gold_program),
        'reference_response': target,
        'source_dataset': source_dataset,
        'task_type': 'program_verifiable',
        'reward_profile': 'program_numeric',
        'record_id': record_id,
        'metadata': {
            'source_path': source_dataset,
            'output_schema': 'Evidence + Program',
            'program_available': True,
            'program_executes': True,
        },
    }


def dedupe_by_contract(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for row in rows:
        key = (
            row['input_prompt_raw'],
            row['gold_program'],
            row['gold_answer'],
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(row)
    return out


def stratified_holdout(rows: List[Dict[str, Any]], valid_n: int, train_n: int, seed: int, label: str) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    rows = sample_rows(rows, None, seed)
    required = valid_n + train_n
    if len(rows) < required:
        raise ValueError(f'{label}: need {required} rows, got {len(rows)}')
    valid_rows = rows[:valid_n]
    train_rows = rows[valid_n:required]
    unused_rows = rows[required:]
    return train_rows, valid_rows, unused_rows


def load_program_rows(path: Path, fallback_source: str) -> List[Dict[str, Any]]:
    source_rows = read_jsonl(path)
    converted = []
    for row in source_rows:
        source_dataset = infer_source_dataset(row, fallback_source)
        item = sharegpt_to_grpo(row, source_dataset)
        if item is not None:
            converted.append(item)
    return dedupe_by_contract(converted)


finqa_candidates = load_program_rows(FINQA_TRAIN_STRICT, 'finqa')
conv_candidates = load_program_rows(SFT2_CONV_STRICT, 'convfinqa_turn')

core_train_finqa_rows, valid_finqa_rows, unused_finqa_rows = stratified_holdout(
    finqa_candidates, VALID_FINQA_ROWS, TRAIN_FINQA_ROWS, SEED + 1, 'finqa'
)
core_train_conv_rows, valid_conv_rows, unused_conv_rows = stratified_holdout(
    conv_candidates, VALID_CONV_ROWS, TRAIN_CONV_ROWS, SEED + 2, 'convfinqa_turn'
)

if EXPAND_TO_FULL_BALANCED_CORE:
    balanced_candidates = load_program_rows(FULL_BALANCED_TRAIN_FILE, 'finqa')
    train_rows = sample_rows(balanced_candidates, None, SEED + 3)
    train_rows = dedupe_by_contract(train_rows)
else:
    train_rows = sample_rows(core_train_conv_rows + core_train_finqa_rows, None, SEED + 3)

valid_rows = sample_rows(valid_conv_rows + valid_finqa_rows, None, SEED + 4)
smoke_rows = sample_rows(core_train_conv_rows, SMOKE_CONV_ROWS, SEED + 5) + sample_rows(core_train_finqa_rows, SMOKE_FINQA_ROWS, SEED + 6)
smoke_rows = sample_rows(smoke_rows, None, SEED + 7)

all_rows = sample_rows(train_rows + valid_rows, None, SEED + 8)

print('FinQA candidates:', len(finqa_candidates), 'train:', len(core_train_finqa_rows), 'valid:', len(valid_finqa_rows), 'unused:', len(unused_finqa_rows))
print('ConvFinQA candidates:', len(conv_candidates), 'train:', len(core_train_conv_rows), 'valid:', len(valid_conv_rows), 'unused:', len(unused_conv_rows))
print('Train rows:', len(train_rows))
print('Valid rows:', len(valid_rows))
print('Smoke rows:', len(smoke_rows))
pd.DataFrame(all_rows[:3])[['source_dataset', 'reward_profile', 'gold_answer', 'gold_program']]


FinQA candidates: 3659 train: 1167 valid: 153 unused: 2339
ConvFinQA candidates: 6369 train: 2333 valid: 307 unused: 3729
Train rows: 3500
Valid rows: 460
Smoke rows: 64


,source_dataset,reward_profile,gold_answer,gold_program
0,finqa,program_numeric,0.4902,"divide(subtract(152, 102), 102)"
1,convfinqa_turn,program_numeric,-0.74163,"subtract(8211, 31780), divide(#0, 31780)"
2,finqa,program_numeric,0.06665,"divide(subtract(1187.4, 1113.2), 1113.2)"


In [15]:
source_files = pd.DataFrame([
    {
        'role': 'finqa_train_strict',
        'path': str(FINQA_TRAIN_STRICT),
        'exists': FINQA_TRAIN_STRICT.exists(),
    },
    {
        'role': 'convfinqa_train_strict',
        'path': str(SFT2_CONV_STRICT),
        'exists': SFT2_CONV_STRICT.exists(),
    },
    {
        'role': 'full_balanced_train_optional',
        'path': str(FULL_BALANCED_TRAIN_FILE),
        'exists': FULL_BALANCED_TRAIN_FILE.exists(),
    },
    {
        'role': 'full_balanced_summary_optional',
        'path': str(FULL_BALANCED_SUMMARY_FILE),
        'exists': FULL_BALANCED_SUMMARY_FILE.exists(),
    },
])
display(source_files)

if FULL_BALANCED_SUMMARY_FILE.exists():
    print(FULL_BALANCED_SUMMARY_FILE.read_text(encoding='utf-8'))


,role,path,exists
0,finqa_train_strict,/root/autodl-tmp/data/financial_reasoning_cot_...,True
1,convfinqa_train_strict,/root/autodl-tmp/data/financial_reasoning_cot_...,True
2,full_balanced_train_optional,/root/autodl-tmp/data/financial_reasoning_v3/c...,True
3,full_balanced_summary_optional,/root/autodl-tmp/data/financial_reasoning_v3/c...,True


{
  "sft_variant": "program_executor_sft",
  "convfinqa_mode": "turn_level",
  "same_prompt_conflicting_labels": 0,
  "sft1_rows": 3686,
  "sft2_convfinqa_source_rows": 6386,
  "sft2_train_convfinqa_rows": 6079,
  "sft2_train_finqa_replay_rows": 3040,
  "sft2_train_rows": 9119,
  "validation_convfinqa_rows": 307,
  "validation_finqa_rows": 153,
  "validation_rows": 460,
  "convfinqa_to_finqa_ratio": 1.9996710526315788,
  "replayed_rows": 0,
  "sft2_convfinqa_only_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_convfinqa_turn_program_strict.jsonl",
  "sft2_finqa_replay_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_finqa_replay_program.jsonl",
  "sft2_balanced_file": "/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced.jsonl",
  "validation_file": "/root/autodl-tmp/data/financial_reasoning_v3/validation/valid_program_balanced.jsonl",
  "train_audit": {
    "rows": 9119,
    "program_parse_success": 1.0,
    "program

In [16]:
print('Program-only GRPO sources:')
print('FinQA strict file:', FINQA_TRAIN_STRICT)
print('ConvFinQA strict file:', SFT2_CONV_STRICT)
print('Optional full balanced train file:', FULL_BALANCED_TRAIN_FILE)
print('Design doc:', DESIGN_DOC)

source_contract = pd.DataFrame([
    {
        'source': 'finqa',
        'reward_profile': 'program_numeric',
        'program_reward': True,
        'main_benchmark': True,
        'role': 'strict Program single-turn verifiable RL',
    },
    {
        'source': 'convfinqa_turn',
        'reward_profile': 'program_numeric',
        'program_reward': True,
        'main_benchmark': True,
        'role': 'strict Program multi-turn verifiable RL',
    },
])

display(source_contract)
assert set(source_contract['reward_profile']) == {'program_numeric'}


Program-only GRPO sources:
FinQA strict file: /root/autodl-tmp/data/financial_reasoning_cot_pot/program_sft/train_finqa_program_strict.jsonl
ConvFinQA strict file: /root/autodl-tmp/data/financial_reasoning_cot_pot/program_sft/train_convfinqa_turn_program_strict.jsonl
Optional full balanced train file: /root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced.jsonl
Design doc: /root/FinQA/run_fingpt_cot_pot.ipynb


,source,reward_profile,program_reward,main_benchmark,role
0,finqa,program_numeric,True,True,strict Program single-turn verifiable RL
1,convfinqa_turn,program_numeric,True,True,strict Program multi-turn verifiable RL


In [17]:
def audit_program_numeric_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    execute_fail = 0
    answer_mismatch = 0
    forbidden_schema = 0
    missing_program = 0
    missing_answer = 0
    bad_reward_profile = 0
    prompt_duplicates = 0
    seen_prompts = set()

    for row in rows:
        prompt = first_text(row.get('input_prompt_raw'))
        if prompt in seen_prompts:
            prompt_duplicates += 1
        seen_prompts.add(prompt)

        if row.get('reward_profile') != 'program_numeric':
            bad_reward_profile += 1

        gold_answer = first_text(row.get('gold_answer'))
        gold_program = first_text(row.get('gold_program'))
        if not gold_answer:
            missing_answer += 1
        if not gold_program:
            missing_program += 1

        executed_value, _, _ = execute_prediction_program(gold_program)
        if executed_value is None:
            execute_fail += 1
        elif not numeric_equal(str(executed_value), gold_answer):
            answer_mismatch += 1

        ref = first_text(row.get('reference_response'))
        if any(anchor in ref for anchor in PROGRAM_NUMERIC_FORBIDDEN):
            forbidden_schema += 1

    return {
        'rows': len(rows),
        'program_execute_fail_rows': execute_fail,
        'program_answer_mismatch_rows': answer_mismatch,
        'forbidden_reference_schema_rows': forbidden_schema,
        'missing_program_rows': missing_program,
        'missing_answer_rows': missing_answer,
        'bad_reward_profile_rows': bad_reward_profile,
        'duplicate_prompt_rows': prompt_duplicates,
    }


def dataset_mix(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).groupby(['source_dataset', 'reward_profile']).size().reset_index(name='rows')


write_jsonl(GRPO_TRAIN_FILE, train_rows)
write_jsonl(GRPO_VALID_FILE, valid_rows)
write_jsonl(GRPO_SMOKE_FILE, smoke_rows)

train_audit = audit_program_numeric_rows(train_rows)
valid_audit = audit_program_numeric_rows(valid_rows)
smoke_audit = audit_program_numeric_rows(smoke_rows)
summary_payload = {
    'seed': SEED,
    'expand_to_full_balanced_core': EXPAND_TO_FULL_BALANCED_CORE,
    'train_rows': len(train_rows),
    'valid_rows': len(valid_rows),
    'smoke_rows': len(smoke_rows),
    'train_conv_rows': sum(1 for row in train_rows if row.get('source_dataset') == 'convfinqa_turn'),
    'train_finqa_rows': sum(1 for row in train_rows if row.get('source_dataset') == 'finqa'),
    'valid_conv_rows': sum(1 for row in valid_rows if row.get('source_dataset') == 'convfinqa_turn'),
    'valid_finqa_rows': sum(1 for row in valid_rows if row.get('source_dataset') == 'finqa'),
    'smoke_conv_rows': sum(1 for row in smoke_rows if row.get('source_dataset') == 'convfinqa_turn'),
    'smoke_finqa_rows': sum(1 for row in smoke_rows if row.get('source_dataset') == 'finqa'),
    'train_audit': train_audit,
    'valid_audit': valid_audit,
    'smoke_audit': smoke_audit,
    'files': {
        'train_file': str(GRPO_TRAIN_FILE),
        'valid_file': str(GRPO_VALID_FILE),
        'smoke_file': str(GRPO_SMOKE_FILE),
    },
}
GRPO_DATA_SUMMARY_FILE.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('train:', len(train_rows), GRPO_TRAIN_FILE)
print('valid:', len(valid_rows), GRPO_VALID_FILE)
print('smoke:', len(smoke_rows), GRPO_SMOKE_FILE)
print('summary:', GRPO_DATA_SUMMARY_FILE)
print('train audit:', train_audit)
print('valid audit:', valid_audit)
print('smoke audit:', smoke_audit)
display(dataset_mix(train_rows))
preview_cols = ['source_dataset', 'reward_profile', 'gold_answer', 'gold_program', 'input_prompt_raw']
display(pd.DataFrame(smoke_rows[:5])[preview_cols])


train: 3500 /root/autodl-tmp/data/financial_reasoning_cot_pot/rl/train_cot_pot_grpo_mixed.jsonl
valid: 460 /root/autodl-tmp/data/financial_reasoning_cot_pot/rl/valid_cot_pot_grpo_mixed.jsonl
smoke: 64 /root/autodl-tmp/data/financial_reasoning_cot_pot/rl/smoke_cot_pot_grpo_mixed.jsonl
summary: /root/autodl-tmp/data/financial_reasoning_cot_pot/rl/grpo_program_only_summary.json
train audit: {'rows': 3500, 'program_execute_fail_rows': 0, 'program_answer_mismatch_rows': 0, 'forbidden_reference_schema_rows': 0, 'missing_program_rows': 0, 'missing_answer_rows': 0, 'bad_reward_profile_rows': 0, 'duplicate_prompt_rows': 0}
valid audit: {'rows': 460, 'program_execute_fail_rows': 0, 'program_answer_mismatch_rows': 0, 'forbidden_reference_schema_rows': 0, 'missing_program_rows': 0, 'missing_answer_rows': 0, 'bad_reward_profile_rows': 0, 'duplicate_prompt_rows': 0}
smoke audit: {'rows': 64, 'program_execute_fail_rows': 0, 'program_answer_mismatch_rows': 0, 'forbidden_reference_schema_rows': 0, 'mis

,source_dataset,reward_profile,rows
0,convfinqa_turn,program_numeric,2333
1,finqa,program_numeric,1167


,source_dataset,reward_profile,gold_answer,gold_program,input_prompt_raw
0,convfinqa_turn,program_numeric,6700,6700,You are a financial conversational reasoning a...
1,convfinqa_turn,program_numeric,2408.8,2408.8,You are a financial conversational reasoning a...
2,convfinqa_turn,program_numeric,-0.18182,"subtract(2.7, 3.3), divide(#0, 3.3)",You are a financial conversational reasoning a...
3,convfinqa_turn,program_numeric,19,"add(10.3, 6.2), add(#0, 2.5)",You are a financial conversational reasoning a...
4,convfinqa_turn,program_numeric,156165,156165,You are a financial conversational reasoning a...


## DPO 数据与训练

DPO 继续保留为主线之一，但它只服务 strict Program 路线，不参与 supplement mixed schema。

DPO contract：



外部 CoT supplement 不进入 DPO pair 构造。DPO 的目标是验证模型能否更稳定地偏好 strict  输出，而不是学习第二套  偏好。


In [ ]:
from financial_data_processors.common import build_rejected_from_strict_response

DPO_TOTAL_BUDGET = 5000
DPO_CONV_TO_FINQA_RATIO = 2.0
DPO_SMOKE_ROWS = 64


def sharegpt_to_dpo(row: Dict[str, Any], source_dataset: str) -> Optional[Dict[str, Any]]:
    conv = row.get('conversations') or []
    if len(conv) < 2:
        return None
    input_prompt_raw = conv[0].get('value') or conv[0].get('content') or ''
    chosen = conv[1].get('value') or conv[1].get('content') or ''
    if not input_prompt_raw or not chosen or 'Program:' not in chosen:
        return None
    rejected = build_rejected_from_strict_response(chosen)
    return {
        'system': '',
        'history': [],
        'question': input_prompt_raw,
        'input_prompt_raw': input_prompt_raw,
        'reward_profile': 'program_numeric',
        'response_chosen': chosen,
        'response_rejected': rejected,
        'source_dataset': source_dataset,
        'record_id': first_text(row.get('record_id') or (row.get('metadata') or {}).get('record_id')),
        'metadata': {
            'program_available': True,
            'base_policy': str(POLICY_BASE),
            'output_schema': 'Evidence + Program',
        },
    }


finqa_dpo = [x for x in (sharegpt_to_dpo(r, 'finqa') for r in read_jsonl(FINQA_TRAIN_STRICT)) if x]
conv_dpo = [x for x in (sharegpt_to_dpo(r, 'convfinqa_turn') for r in read_jsonl(SFT2_CONV_STRICT)) if x]

conv_target = int(round(DPO_TOTAL_BUDGET * DPO_CONV_TO_FINQA_RATIO / (DPO_CONV_TO_FINQA_RATIO + 1)))
finqa_target = DPO_TOTAL_BUDGET - conv_target
dpo_rows = sample_rows(conv_dpo, conv_target, SEED + 31) + sample_rows(finqa_dpo, finqa_target, SEED + 32)
dpo_rows = sample_rows(dpo_rows, None, SEED + 33)
dpo_smoke_rows = sample_rows(dpo_rows, DPO_SMOKE_ROWS, SEED + 34)

write_jsonl(DPO_MIXED_FILE, dpo_rows)
write_jsonl(DPO_SMOKE_FILE, dpo_smoke_rows)
(DPO_TRAIN_DIR / DPO_MIXED_FILE.name).write_text(DPO_MIXED_FILE.read_text(encoding='utf-8'), encoding='utf-8')
(DPO_SMOKE_DIR / DPO_SMOKE_FILE.name).write_text(DPO_SMOKE_FILE.read_text(encoding='utf-8'), encoding='utf-8')

print({
    'finqa_dpo_candidates': len(finqa_dpo),
    'conv_dpo_candidates': len(conv_dpo),
    'dpo_rows': len(dpo_rows),
    'dpo_file': str(DPO_MIXED_FILE),
    'dpo_smoke_rows': len(dpo_smoke_rows),
    'reward_profile': 'program_numeric',
})
display(pd.DataFrame(dpo_rows[:3])[['source_dataset', 'record_id', 'reward_profile', 'response_chosen', 'response_rejected']])


In [ ]:
DPO_MAX_STEPS = 120
DPO_SMOKE_MAX_STEPS = 1
DPO_LEARNING_RATE = '5e-6'
DPO_MAX_SOURCE_LENGTH = '1536'
DPO_MAX_TARGET_LENGTH = '256'

print('Use the explicit !python cells below to run DPO smoke or full training.')
print({
    'model_name_or_path': str(POLICY_BASE),
    'tokenizer_name_or_path': str(BASE_MODEL),
    'dpo_smoke_train_dir': str(DPO_SMOKE_DIR),
    'dpo_full_train_dir': str(DPO_TRAIN_DIR),
    'dpo_smoke_output_dir': str(DPO_SMOKE_OUT),
    'dpo_full_output_dir': str(DPO_OUT),
    'dpo_smoke_max_steps': DPO_SMOKE_MAX_STEPS,
    'dpo_full_max_steps': DPO_MAX_STEPS,
})


### DPO Commands

In [ ]:
# DPO smoke
!python -m training.dpo_training \
  --model_name_or_path {POLICY_BASE} \
  --tokenizer_name_or_path {BASE_MODEL} \
  --template_name qwen \
  --validation_split_percentage 1 \
  --eval_strategy no \
  --train_file_dir {DPO_SMOKE_DIR} \
  --do_train \
  --use_peft True \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --gradient_checkpointing True \
  --learning_rate {DPO_LEARNING_RATE} \
  --max_steps {DPO_SMOKE_MAX_STEPS} \
  --max_source_length {DPO_MAX_SOURCE_LENGTH} \
  --max_target_length {DPO_MAX_TARGET_LENGTH} \
  --logging_steps 10 \
  --save_steps 40 \
  --logging_first_step True \
  --target_modules q_proj,k_proj,v_proj,o_proj \
  --lora_rank 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05 \
  --torch_dtype bfloat16 \
  --device_map auto \
  --ddp_find_unused_parameters False \
  --output_dir {DPO_SMOKE_OUT}


In [ ]:
# DPO full
!python -m training.dpo_training \
  --model_name_or_path {POLICY_BASE} \
  --tokenizer_name_or_path {BASE_MODEL} \
  --template_name qwen \
  --validation_split_percentage 1 \
  --eval_strategy no \
  --train_file_dir {DPO_TRAIN_DIR} \
  --do_train \
  --use_peft True \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --gradient_checkpointing True \
  --learning_rate {DPO_LEARNING_RATE} \
  --max_steps {DPO_MAX_STEPS} \
  --max_source_length {DPO_MAX_SOURCE_LENGTH} \
  --max_target_length {DPO_MAX_TARGET_LENGTH} \
  --logging_steps 10 \
  --save_steps 40 \
  --logging_first_step True \
  --target_modules q_proj,k_proj,v_proj,o_proj \
  --lora_rank 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05 \
  --torch_dtype bfloat16 \
  --device_map auto \
  --ddp_find_unused_parameters False \
  --output_dir {DPO_OUT}


## Reward 函数

Reward contract 现在按 `reward_profile` 显式分流：

| reward_profile | 适用数据 | 主 schema | 主要 reward |
|---|---|---|---|
| `program_numeric` | FinQA / ConvFinQA | `Evidence + Program` | program execution、strict format、program op overlap、evidence grounding、轻量长度约束 |
| `cot_answer_only` | Fino1 / FinCoT | `Reasoning + Answer` | answer correctness、CoT format、简洁性、避免复述负样本回答 |

约束：

- `program_numeric` 不允许输出 `Reasoning:`、`Answer:`、`Normalized Answer:`。
- `cot_answer_only` 不要求 Program，也不应该被 Program reward 干扰。
- evidence-level reward 只读取 `input_prompt_raw`，不读取 chat list 的字符串化结果。
- 错误但可执行的 Program 不再获得保底正奖励。


In [18]:
from evaluation.evaluate_financial_benchmarks import execute_prediction_program


def reward_answer(completions, gold_answer=None, **kwargs):
    rewards = []
    gold_answer = gold_answer or [''] * len(completions)
    for completion, gold in zip(completions, gold_answer):
        text = completion_text(completion)
        program = extract_anchor(text, 'Program:')
        executed_value, _, _ = execute_prediction_program(program)
        rewards.append(0.55 if executed_value is not None and numeric_equal(str(executed_value), gold) else 0.0)
    return rewards


def reward_format(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion_text(completion)
        score = sum(1 for anchor in PROGRAM_NUMERIC_REQUIRED if anchor in text) / len(PROGRAM_NUMERIC_REQUIRED)
        penalty = sum(1 for anchor in PROGRAM_NUMERIC_FORBIDDEN if anchor in text) * 0.15
        rewards.append(max(0.0, score * 0.18 - penalty))
    return rewards


def reward_program(completions, gold_program=None, **kwargs):
    rewards = []
    gold_program = gold_program or [''] * len(completions)
    for completion, gold in zip(completions, gold_program):
        text = completion_text(completion)
        pred_program = extract_anchor(text, 'Program:')
        gold_ops = program_ops(gold)
        pred_ops = program_ops(pred_program)
        if not pred_program or pred_program.strip().upper() == 'N/A':
            rewards.append(0.0)
            continue
        executed_value, _, _ = execute_prediction_program(pred_program)
        parse_bonus = 0.08 if executed_value is not None else 0.0
        op_bonus = 0.0
        if gold_ops and pred_ops:
            op_bonus = sum(1 for op in set(gold_ops) if op in set(pred_ops)) / len(set(gold_ops)) * 0.12
        rewards.append(parse_bonus + op_bonus)
    return rewards


def reward_program_answer_consistency(completions, gold_answer=None, **kwargs):
    rewards = []
    gold_answer = gold_answer or [''] * len(completions)
    for completion, gold in zip(completions, gold_answer):
        program = extract_anchor(completion_text(completion), 'Program:')
        executed_value, _, _ = execute_prediction_program(program)
        if executed_value is None:
            rewards.append(0.0)
        elif numeric_equal(str(executed_value), gold):
            rewards.append(0.10)
        else:
            rewards.append(0.0)
    return rewards


def reward_evidence(completions, input_prompt_raw=None, prompt=None, **kwargs):
    rewards = []
    input_prompt_raw = input_prompt_raw or prompt or [''] * len(completions)
    for completion, raw_prompt in zip(completions, input_prompt_raw):
        evidence = extract_anchor(completion_text(completion), 'Evidence:')
        raw_prompt_text = prompt_text_from_any(raw_prompt)
        ev_nums = set(NUMBER_RE.findall(evidence.replace(',', '')))
        prompt_nums = set(NUMBER_RE.findall(raw_prompt_text.replace(',', '')))
        rewards.append(0.05 if ev_nums and ev_nums.intersection(prompt_nums) else 0.0)
    return rewards


def reward_brevity_and_relevance(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion_text(completion)
        words = text.split()
        rewards.append(0.05 if 8 <= len(words) <= 120 else 0.0)
    return rewards


REWARD_FUNCS = [
    reward_answer,
    reward_format,
    reward_program,
    reward_program_answer_consistency,
    reward_evidence,
    reward_brevity_and_relevance,
]
print([fn.__name__ for fn in REWARD_FUNCS])


['reward_answer', 'reward_format', 'reward_program', 'reward_program_answer_consistency', 'reward_evidence', 'reward_brevity_and_relevance']


In [19]:
smoke_completions = [
    'Evidence:\n- the 2007 value is 991.1 and the 2008 value is 959.2\n- compute the year-over-year change\n\nProgram: divide(subtract(959.2, 991.1), 991.1)',
    'Evidence:\n- the values are listed above\n\nProgram: add(1, 1)',
    'Reasoning:\nUse the change ratio.\n\nProgram: divide(subtract(959.2, 991.1), 991.1)',
    'Evidence:\n- revenue was 20\n- cost was 15\n\nProgram: subtract(20, 15)\nadd(0, 0)',
]
smoke_kwargs = {
    'gold_answer': ['-0.03219', '-0.03219', '-0.03219', '5'],
    'gold_program': [
        'divide(subtract(959.2, 991.1), 991.1)',
        'divide(subtract(959.2, 991.1), 991.1)',
        'divide(subtract(959.2, 991.1), 991.1)',
        'subtract(20, 15)',
    ],
    'reward_profile': ['program_numeric', 'program_numeric', 'program_numeric', 'program_numeric'],
    'input_prompt_raw': [
        '2007 991.1 2008 959.2',
        '2007 991.1 2008 959.2',
        '2007 991.1 2008 959.2',
        'Revenue 20 cost 15',
    ],
}
for fn in REWARD_FUNCS:
    print(fn.__name__, fn(smoke_completions, **smoke_kwargs))


reward_answer [0.55, 0.0, 0.55, 0.0]
reward_format [0.18, 0.18, 0.0, 0.18]
reward_program [0.2, 0.08, 0.2, 0.12]
reward_program_answer_consistency [0.1, 0.0, 0.1, 0.0]
reward_evidence [0.05, 0.0, 0.0, 0.05]
reward_brevity_and_relevance [0.05, 0.05, 0.05, 0.05]


## GRPO 训练

这一节不再在 notebook 内部直接启动 GRPOTrainer。

数据处理完成后，直接运行下面的 shell 脚本即可开始训练：

- `bash /root/FinQA/run_finqa_program_grpo_base.sh`
- `bash /root/FinQA/run_finqa_program_grpo_v3.sh`

这样不需要为每次训练重新跑完整 notebook。


In [20]:
GRPO_NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 384
GRPO_LEARNING_RATE = 5e-6
BETA = 0.001
MAX_STEPS = 300
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

BASE_GRPO_OUT = grpo_output_dir('base', smoke=False)
V3_GRPO_OUT = grpo_output_dir('v3_program_sft', smoke=False)

print({
    'training_script': str(TRAINING_SCRIPT),
    'base_grpo_script': str(BASE_GRPO_SCRIPT),
    'v3_grpo_script': str(V3_GRPO_SCRIPT),
    'train_file': str(GRPO_TRAIN_FILE),
    'valid_file': str(GRPO_VALID_FILE),
    'base_output_dir': str(BASE_GRPO_OUT),
    'v3_output_dir': str(V3_GRPO_OUT),
})


{'GRPO_NUM_GENERATIONS': 4, 'MAX_PROMPT_LENGTH': 1536, 'MAX_COMPLETION_LENGTH': 384, 'GRPO_LEARNING_RATE': 5e-06, 'BETA': 0.001, 'MAX_STEPS': 300, 'USE_VLLM': False, 'RUN_GRPO_SMOKE_EXPERIMENT_KEY': 'cot_pot_program_mixed', 'RUN_FULL_GRPO_EXPERIMENT_KEYS': ['base', 'v3_program_sft', 'cot_pot_program_mixed'], 'train_file': '/root/autodl-tmp/data/financial_reasoning_cot_pot/rl/train_cot_pot_grpo_mixed.jsonl', 'valid_file': '/root/autodl-tmp/data/financial_reasoning_cot_pot/rl/valid_cot_pot_grpo_mixed.jsonl'}


In [ ]:
print(BASE_GRPO_SCRIPT.read_text(encoding='utf-8'))
print('\n---\n')
print(V3_GRPO_SCRIPT.read_text(encoding='utf-8'))

print('\nRun directly in terminal or notebook cells:')
print('bash', BASE_GRPO_SCRIPT)
print('bash', V3_GRPO_SCRIPT)


## Benchmark 命令

主 benchmark 现在按实验分别跑：

- `baseline`：只评估当前起点 policy
- `grpo`：在同一 policy 上挂载本 notebook 产出的 GRPO adapter

这样每个实验都能回答两个问题：

1. 这个起点本身有多强。
2. 在这个起点上继续做 mixed GRPO，strict Program 主指标到底有没有提升。

主 benchmark 仍严格对齐 `run_fingpt_cot_pot.ipynb`：

- 评测口径 = strict Program
- processor = `program_executor_sft`
- numeric output format = `cot_program`


In [ ]:
BENCHMARK_EXPERIMENT_KEYS = ['base', 'v3_program_sft']
RUN_BENCHMARK_SUPPLEMENT_DIAG = False

print('Use the explicit !python benchmark cells below to run baseline or GRPO evaluation for each experiment key.')
print({
    'benchmark_experiment_keys': BENCHMARK_EXPERIMENT_KEYS,
    'tokenizer_path': str(BASE_MODEL),
    'finqa_test_file': str(FINQA_EVAL_FILE),
    'convfinqa_test_file': str(CONVFINQA_EVAL_FILE),
})
if RUN_BENCHMARK_SUPPLEMENT_DIAG:
    print('Supplement diagnostics are intentionally kept out of the strict Program main benchmark table.')
else:
    print('Supplement diag skipped. Keep main benchmark strict-program only.')


### Benchmark Commands

下面这些 cell 直接用 `!python -m evaluation.evaluate_financial_benchmarks ...`。按实验逐个运行即可。


In [ ]:
# baseline benchmark: base
EXPERIMENT_KEY = 'base'
SPEC = EXPERIMENT_SPECS[EXPERIMENT_KEY]
BASELINE_OUT = benchmark_output_dir(EXPERIMENT_KEY, 'baseline_passk')
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path {BASE_MODEL} \
  --model_entry baseline={SPEC['policy_benchmark_path']} \
  --finqa_test_file {FINQA_EVAL_FILE} \
  --convfinqa_test_file {CONVFINQA_EVAL_FILE} \
  --finqa_max_samples 100 \
  --convfinqa_max_samples 100 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --numeric_output_format cot_program \
  --output_dir {BASELINE_OUT}


In [ ]:
# grpo benchmark: base
EXPERIMENT_KEY = 'base'
SPEC = EXPERIMENT_SPECS[EXPERIMENT_KEY]
GRPO_ADAPTER = grpo_output_dir(EXPERIMENT_KEY, smoke=False)
GRPO_OUT_DIR = benchmark_output_dir(EXPERIMENT_KEY, 'grpo_passk')
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path {BASE_MODEL} \
  --model_entry baseline={SPEC['policy_benchmark_path']} \
  --model_entry grpo={SPEC['policy_benchmark_path']} \
  --adapter_entry grpo={GRPO_ADAPTER} \
  --finqa_test_file {FINQA_EVAL_FILE} \
  --convfinqa_test_file {CONVFINQA_EVAL_FILE} \
  --finqa_max_samples 100 \
  --convfinqa_max_samples 100 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --numeric_output_format cot_program \
  --output_dir {GRPO_OUT_DIR}


In [ ]:
# baseline benchmark: v3_program_sft
EXPERIMENT_KEY = 'v3_program_sft'
SPEC = EXPERIMENT_SPECS[EXPERIMENT_KEY]
BASELINE_OUT = benchmark_output_dir(EXPERIMENT_KEY, 'baseline_passk')
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path {BASE_MODEL} \
  --model_entry baseline={SPEC['policy_benchmark_path']} \
  --finqa_test_file {FINQA_EVAL_FILE} \
  --convfinqa_test_file {CONVFINQA_EVAL_FILE} \
  --finqa_max_samples 100 \
  --convfinqa_max_samples 100 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --numeric_output_format cot_program \
  --output_dir {BASELINE_OUT}


In [ ]:
# grpo benchmark: v3_program_sft
EXPERIMENT_KEY = 'v3_program_sft'
SPEC = EXPERIMENT_SPECS[EXPERIMENT_KEY]
GRPO_ADAPTER = grpo_output_dir(EXPERIMENT_KEY, smoke=False)
GRPO_OUT_DIR = benchmark_output_dir(EXPERIMENT_KEY, 'grpo_passk')
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path {BASE_MODEL} \
  --model_entry baseline={SPEC['policy_benchmark_path']} \
  --model_entry grpo={SPEC['policy_benchmark_path']} \
  --adapter_entry grpo={GRPO_ADAPTER} \
  --finqa_test_file {FINQA_EVAL_FILE} \
  --convfinqa_test_file {CONVFINQA_EVAL_FILE} \
  --finqa_max_samples 100 \
  --convfinqa_max_samples 100 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --processor_sft_variant program_executor_sft \
  --numeric_output_format cot_program \
  --output_dir {GRPO_OUT_DIR}


In [ ]:
# removed: cot_pot_program_mixed baseline benchmark
print('cot_pot_program_mixed benchmark removed in this version. Only base and v3_program_sft are trained/evaluated.')


In [ ]:
# removed: cot_pot_program_mixed grpo benchmark
print('cot_pot_program_mixed benchmark removed in this version. Only base and v3_program_sft are trained/evaluated.')


## 结果分析

这一节只比较两条线：

1. `base` vs `base + GRPO`
2. `v3_program_sft` vs `v3_program_sft + GRPO`

口径约束保持不变：

- 主表只看 strict Program benchmark
- 数据集仍是 `program_numeric-only`
- legacy summary 只作为参考，不覆盖本 notebook 当前跑出来的 benchmark


In [ ]:
def reward_profile_mix_summary(path: Path) -> Dict[str, int]:
    rows = read_jsonl(path)
    return {
        'program_numeric_rows': sum(1 for row in rows if row.get('reward_profile') == 'program_numeric'),
    }


def load_summary_frame(path: Path, experiment_key: str, run_kind: str) -> Optional[pd.DataFrame]:
    if not path.exists():
        print('missing summary:', experiment_key, run_kind, path)
        return None
    df = pd.read_csv(path)
    df.insert(0, 'experiment_key', experiment_key)
    df.insert(1, 'run_kind', run_kind)
    df.insert(2, 'label', EXPERIMENT_SPECS[experiment_key]['label'])
    return df


summary_frames = []
for key in EXPERIMENT_ORDER:
    for run_kind, summary_dir in [
        ('baseline', benchmark_output_dir(key, 'baseline_passk')),
        ('grpo', benchmark_output_dir(key, 'grpo_passk')),
    ]:
        df = load_summary_frame(summary_dir / 'benchmark_summary.csv', key, run_kind)
        if df is not None:
            summary_frames.append(df)

legacy_frames = []
for key in EXPERIMENT_ORDER:
    legacy_path = EXPERIMENT_SPECS[key]['legacy_summary']
    if legacy_path is None or not legacy_path.exists():
        continue
    df = pd.read_csv(legacy_path)
    df.insert(0, 'experiment_key', key)
    df.insert(1, 'run_kind', 'legacy_reference')
    df.insert(2, 'label', EXPERIMENT_SPECS[key]['label'])
    legacy_frames.append(df)

mix_counts = reward_profile_mix_summary(GRPO_TRAIN_FILE) if GRPO_TRAIN_FILE.exists() else {'program_numeric_rows': 0}
print('reward_profile_mix:', mix_counts)

if summary_frames:
    current_summary = pd.concat(summary_frames, ignore_index=True)
    current_summary['reward_profile_mix'] = f"program_numeric={mix_counts['program_numeric_rows']}"
    display(current_summary)
else:
    print('No current benchmark summaries available yet.')

if legacy_frames:
    legacy_summary = pd.concat(legacy_frames, ignore_index=True)
    print('legacy references:')
    display(legacy_summary)
else:
    print('No legacy summaries available.')


In [ ]:
def load_greedy_predictions(path: Path) -> Dict[Tuple[str, str], Dict[str, Any]]:
    rows = {}
    if not path.exists():
        return rows
    for row in read_jsonl(path):
        if row.get('generation_mode') == 'greedy':
            rows[(row.get('task_name', ''), row.get('record_id', ''))] = row
    return rows


for experiment_key in EXPERIMENT_ORDER:
    baseline_path = benchmark_output_dir(experiment_key, 'baseline_passk') / 'baseline_predictions.jsonl'
    grpo_path = benchmark_output_dir(experiment_key, 'grpo_passk') / 'grpo_predictions.jsonl'

    baseline_preds = load_greedy_predictions(baseline_path)
    grpo_preds = load_greedy_predictions(grpo_path)
    common_keys = sorted(set(baseline_preds).intersection(grpo_preds))

    print('\n=== greedy comparison:', experiment_key, '===')
    print('baseline predictions:', len(baseline_preds), baseline_path)
    print('grpo predictions:', len(grpo_preds), grpo_path)
    print('common greedy predictions:', len(common_keys))

    for key in common_keys[:5]:
        base_row = baseline_preds[key]
        grpo_row = grpo_preds[key]
        print('\n', key)
        print('baseline correct:', base_row.get('answer_correct'), 'grpo correct:', grpo_row.get('answer_correct'))
        print('gold:', base_row.get('gold_answer'))
        print('baseline:', first_text(base_row.get('prediction'))[:500])
        print('grpo:', first_text(grpo_row.get('prediction'))[:500])


## 扩展说明

这一版 notebook 先把 **program_numeric-only GRPO core** 固定下来：

- 训练数据 `3500` 条
- `ConvFinQA : FinQA = 2333 : 1167`
- 验证集固定 `307 : 153`
- smoke 集固定 `43 : 21`

当前只训练两条线：

- `base + GRPO`
- `v3_program_sft + GRPO`

如果第一阶段稳定，再考虑扩到更多起点或更多数据，但前提是这些起点必须有真实存在、可直接加载的本地模型目录。
